In [2]:
!pip install nltk scikit-learn

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import nltk
import re

from collections import Counter
from nltk.corpus import movie_reviews
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
documents = []
labels = []

for file_id in movie_reviews.fileids():
    review_text = movie_reviews.raw(file_id)
    sentiment = movie_reviews.categories(file_id)[0]

    documents.append(review_text)
    labels.append(1 if sentiment == 'pos' else 0)

print("Total reviews:", len(documents))
print("Total labels:", len(labels))
print("First review:\\n", documents[0][:500])
print("First label:", labels[0])

Total reviews: 2000
Total labels: 2000
First review:\n plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt
First label: 0


In [6]:
for i in range(5):
    print(f"Review {i+1}:")
    print(documents[i])
    print("Label:", labels[i])
    print("-" * 100)

Review 1:
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn't snag this one correctly . 
they seem to have taken this pretty neat concept , but executed it terribly . 
so what are the problems with the movie ? 
well , its main problem is that it's simply too jumbled . 
it starts off " normal " but then downshifts into this " fantasy " world in which you , as an audience member , 

In [7]:
train_documents, test_documents, train_labels, test_labels = train_test_split(
    documents,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("Training reviews:", len(train_documents))
print("Testing reviews:", len(test_documents))


Training reviews: 1600
Testing reviews: 400


In [8]:
train_tokens = [word_tokenize(review.lower()) for review in train_documents]
test_tokens = [word_tokenize(review.lower()) for review in test_documents]

print(train_tokens[0][:30])


['saving', 'private', 'ryan', '(', 'dreamworks', ')', 'running', 'time', ':', '2', 'hours', '48', 'minutes', '.', 'starring', 'tom', 'hanks', ',', 'edward', 'burns', ',', 'tom', 'sizemore', 'and', 'matt', 'damon', 'directed', 'by', 'steven', 'spielberg']


In [9]:
word_counts = Counter()

for review_tokens in train_tokens:
    word_counts.update(review_tokens)

vocab = {
    '<pad>': 0,
    '<unk>': 1
}

min_frequency = 2
max_vocab_size = 20000

for word, frequency in word_counts.most_common():
    if frequency < min_frequency:
        continue
    if len(vocab) >= max_vocab_size:
        break
    if word not in vocab:
        vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:20])

Vocabulary size: 20000
[('<pad>', 0), ('<unk>', 1), (',', 2), ('the', 3), ('.', 4), ('a', 5), ('and', 6), ('of', 7), ('to', 8), ('is', 9), ('in', 10), ("'s", 11), ('``', 12), ('it', 13), ('that', 14), (')', 15), ('(', 16), ('as', 17), ('with', 18), ('for', 19)]


In [9]:
len(vocab)

20000

In [10]:
def text_to_indices(tokens, vocab):
    numerical_review = []

    for token in tokens:
        if token in vocab:
            numerical_review.append(vocab[token])
        else:
            numerical_review.append(vocab['<unk>'])

    return numerical_review



In [11]:
train_numerical_reviews = []

for review_tokens in train_tokens:
    train_numerical_reviews.append(text_to_indices(review_tokens, vocab))

test_numerical_reviews = []

for review_tokens in test_tokens:
    test_numerical_reviews.append(text_to_indices(review_tokens, vocab))

print(train_numerical_reviews[0][:30])

[1222, 889, 692, 16, 3038, 15, 444, 73, 60, 369, 518, 9464, 247, 4, 817, 521, 1604, 2, 1397, 3804, 2, 521, 9465, 6, 1398, 2011, 431, 29, 1038, 1118]


In [12]:
max_len = 300
print("Maximum sequence length:", max_len)

Maximum sequence length: 300


In [13]:
def pad_sequences(sequences, max_len):
    padded_sequences = []

    for sequence in sequences:
        sequence = sequence[:max_len]

        padding_length = max_len - len(sequence)
        padded_sequence = [vocab['<pad>']] * padding_length + sequence

        padded_sequences.append(padded_sequence)

    return torch.tensor(padded_sequences, dtype=torch.long)

X_train = pad_sequences(train_numerical_reviews, max_len)
X_test = pad_sequences(test_numerical_reviews, max_len)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: torch.Size([1600, 300])
X_test shape: torch.Size([400, 300])


In [15]:
y_train = torch.tensor(train_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("First five labels:", y_train[:5])

y_train shape: torch.Size([1600])
y_test shape: torch.Size([400])
First five labels: tensor([1, 1, 0, 0, 1])


In [16]:
print("First input:", X_train[0])
print("First label:", y_train[0].item())

First input: tensor([ 1222,   889,   692,    16,  3038,    15,   444,    73,    60,   369,
          518,  9464,   247,     4,   817,   521,  1604,     2,  1397,  3804,
            2,   521,  9465,     6,  1398,  2011,   431,    29,  1038,  1118,
          495,   122,  7054,    17,     3, 18455,   401,    33,   211,   150,
            2,    89,  1222,   889,   692,     9,    31,  5953,     2, 13051,
            6,  4717,    33,     4,    13,   183,    32,    30,     3,  1039,
          401,    33,    10,   119,  1265,     2,    25,    13,    11,   373,
           35,     7,     3,   126,   401,   132,   150,     2,     6,    35,
            7,     3,   126,     7,  1109,     4,   521,  1604,   388,    17,
            5,  1312,    34,    11, 18456,    42,     8,   199,   889,   692,
           16,  2011,    15,    34,    42,     5,  2814,   278,    97,    21,
          227,   732,    39,    82,   665,    10,   140,     4,   140,     2,
          494,     6,    68,    61,  1661,  2252,  

In [17]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [18]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

print("Training dataset size:", len(train_dataset))
print("Testing dataset size:", len(test_dataset))

Training dataset size: 1600
Testing dataset size: 400


In [19]:
batch_size = 32

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [20]:
for batch_x, batch_y in train_dataloader:
    print("Input batch shape:", batch_x.shape)
    print("Label batch shape:", batch_y.shape)
    print("First five labels:", batch_y[:5])
    break

Input batch shape: torch.Size([32, 300])
Label batch shape: torch.Size([32])
First five labels: tensor([1, 0, 0, 0, 1])


In [21]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("One review:", X_train[0].shape)
print("One review with batch dimension:", X_train[0].unsqueeze(0).shape)

X_train: torch.Size([1600, 300])
y_train: torch.Size([1600])
One review: torch.Size([300])
One review with batch dimension: torch.Size([1, 300])


In [22]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_size=128, num_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=vocab['<pad>']
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)

        output, (hidden_state, cell_state) = self.lstm(embedded)

        final_hidden_state = hidden_state[-1]

        output = self.fc(final_hidden_state)

        return output

In [23]:
model = LSTMModel(vocab_size=len(vocab))
print(model)

LSTMModel(
  (embedding): Embedding(20000, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = model.to(device)

Using device: cuda


In [25]:
sample_input = X_train[:4].to(device)
sample_output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", sample_output.shape)
print("Output scores:", sample_output)

Input shape: torch.Size([4, 300])
Output shape: torch.Size([4, 2])
Output scores: tensor([[ 0.0335,  0.0304],
        [-0.0375,  0.0211],
        [ 0.1384,  0.0708],
        [ 0.1436,  0.0846]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [26]:
epochs = 8
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        output = model(batch_x)
        loss = criterion(output, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_dataloader)
    print(f"Epoch: {epoch + 1}/{epochs}, Loss: {average_loss:.4f}")

Epoch: 1/8, Loss: 0.6935
Epoch: 2/8, Loss: 0.6569
Epoch: 3/8, Loss: 0.5865
Epoch: 4/8, Loss: 0.4620
Epoch: 5/8, Loss: 0.2919
Epoch: 6/8, Loss: 0.1297
Epoch: 7/8, Loss: 0.0483
Epoch: 8/8, Loss: 0.0199


In [27]:
def prediction(model, vocab, text, max_len=300):
    model.eval()

    tokens = word_tokenize(text.lower())
    numerical_text = text_to_indices(tokens, vocab)

    numerical_text = numerical_text[:max_len]
    padding_length = max_len - len(numerical_text)

    padded_text = (
        [vocab['<pad>']] * padding_length + numerical_text
    )

    input_tensor = torch.tensor(
        [padded_text],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = torch.argmax(output, dim=1).item()

    if predicted_class == 1:
        return "Positive"
    else:
        return "Negative"

In [28]:
review = "This movie was wonderful. The acting was excellent and I really enjoyed it."

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: This movie was wonderful. The acting was excellent and I really enjoyed it.
Predicted sentiment: Negative


In [29]:
review = "The story was boring, the acting was poor, and I hated this movie."

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: The story was boring, the acting was poor, and I hated this movie.
Predicted sentiment: Negative


In [30]:
review = "The movie was fantastic! The actors delivered excellent performances, and the story kept me interested until the end."

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: The movie was fantastic! The actors delivered excellent performances, and the story kept me interested until the end.
Predicted sentiment: Positive


In [31]:
review = "I found this movie disappointing. The plot was confusing, the characters were boring, and the ending was terrible."

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: I found this movie disappointing. The plot was confusing, the characters were boring, and the ending was terrible.
Predicted sentiment: Positive


In [32]:
review = "The acting is not good"

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: The acting is not good
Predicted sentiment: Negative


In [33]:
review = "the woman role is not satisfied ."

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: the woman role is not satisfied .
Predicted sentiment: Negative


In [34]:
review = "I feel motive from this movie. This is great"

result = prediction(model, vocab, review)

print("Review:", review)
print("Predicted sentiment:", result)

Review: I feel motive from this movie. This is great
Predicted sentiment: Negative


In [35]:
def calculate_accuracy(model, dataloader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            output = model(batch_x)
            predicted = torch.argmax(output, dim=1)

            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = (correct / total) * 100
    return accuracy

In [36]:
test_accuracy = calculate_accuracy(model, test_dataloader, device)

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 52.25%
